In [ ]:
! pip install orchestrator

In [ ]:
from orchestrator.models import MetaModel

metamodel = MetaModel.model_validate(
    {
        "ran": {
            "components": {
                "gnb": {
                    "type": "gnb",
                },
                "cucp": {
                    "type": "cu_cp",
                    "config": {
                        "cu_up": "cuup",
                        "du": "du2",
                    } 
                },
                "cuup": {
                    "type": "cu_up",
                    "config": {
                        "cu_cp": "cucp",
                        "du": "du2",
                    } 
                },
                "cu1": {
                    "type": "cu",
                    "config": {
                        "du": "du1",
                    }
                },
                "du1": {
                    "type": "du",
                    "config": {
                        "cu_up": "cu1",
                        "cu_cp": "cu1",
                    }
                },
                "du2": {
                    "type": "du",
                    "config": {
                        "cu_up": "cuup",
                        "cu_cp": "cucp",
                    }
                },
            }
        }
    }
)
metamodel.to_yaml_file("metamodel_1.yaml")
metamodel.to_yaml_file("metamodel_full_1.yaml", exclude_defaults=False)

In [2]:
from orchestrator.models import DeploymentModel

deployment_model = DeploymentModel.model_validate(
    {
        "params": {
            "provisioning": {
                "experiment": "tst" # slices experiment
            }
        }
    })
deployment_model.to_yaml_file("deployment_model_1.yaml")
deployment_model.to_yaml_file("deployment_model_full_1.yaml", exclude_defaults=False)

In [3]:
from orchestrator.deployment import BaseBlueprintManagement

reconciled_metamodel, reconciled_deployment_model = BaseBlueprintManagement(metamodel=metamodel, deployment_model=deployment_model).output()

In [4]:
reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_1.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_1.yaml")

In [5]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [6]:
await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/node/bi/provision : Select project 'post5g-beta'] ***********

TASK [provisioning/node/bi/provision : Select project 'post5g-beta'] ***********

In [7]:
metamodel, deployment_model = prov.output()

deployment_model.to_yaml_file("deployment_model_2.yaml")
metamodel.to_yaml_file("metamodel_2.yaml")

# checkpoint 1: provisioned

In [1]:
from orchestrator.deployment import ProvisioningManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_2.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_2.yaml")

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [2]:
prov.get_merged(output="yaml", path="merged_model.yaml")

In [3]:
prov.get_ansible_inventory(path="inventory.ini")

In [4]:
from orchestrator.deployment import K8SSetupManagement

k8s = K8SSetupManagement(metamodel=metamodel, deployment_model=deployment_model)

In [5]:
await k8s.deploy()


PLAY [node2] *******************************************************************

PLAY [node3] *******************************************************************

PLAY [node1] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************
TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************


TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************
[WARNING]: Host 'node3' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node3][WARNING]: Host 'node2' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpre

In [6]:
metamodel, deployment_model = k8s.output()

deployment_model.to_yaml_file("deployment_model_3.yaml")
metamodel.to_yaml_file("metamodel_3.yaml")

# checkpoint 2: k8s ready

In [1]:
from orchestrator.deployment import CoreManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_3.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_3.yaml")

core = CoreManagement(metamodel=metamodel, deployment_model=deployment_model)

In [2]:
reconciled_metamodel, reconciled_deployment_model = core.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_2.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_2.yaml")

In [3]:
core.mutatable_metamodel.to_yaml_file("mutatated_metamodel.yaml", exclude_defaults=False)

In [4]:
await core.deploy()


PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-smf' chart] **********

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-database' chart] *****

TASK [helm/prepare : Create Helm build directory for 'oai-nrf' chart] **********

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-amf' chart] **********

TASK [helm/prepare : Create Helm build directory for 'oai-upf' chart] **********
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpret

In [5]:
metamodel, deployment_model = core.output()

deployment_model.to_yaml_file("deployment_model_4.yaml")
metamodel.to_yaml_file("metamodel_4.yaml")

# checkpoint 3: core deployed

In [1]:
from orchestrator.deployment import RANManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_4.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_4.yaml")

ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

In [2]:
reconciled_metamodel, reconciled_deployment_model = ran.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_3.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_3.yaml")

In [3]:
await ran.deploy()


PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du1' chart] **********

TASK [helm/prepare : Create Helm build directory for 'oai-cuup' chart] *********

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du2' chart] **********

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cu1' chart] **********

TASK [helm/prepare : Create Helm build directory for 'oai-cucp' chart] *********
[WARNING]: Host 

In [4]:
metamodel, deployment_model = ran.output()

deployment_model.to_yaml_file("deployment_model_5.yaml")
metamodel.to_yaml_file("metamodel_5.yaml")

# checkpoint 4: ran deployed

In [1]:
from orchestrator.deployment import RANManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_5.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_5.yaml")

ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

In [4]:
await ran.remove(force=True)


PLAY [node1] *******************************************************************
PLAY [node1] *******************************************************************


TASK [helm/uninstall : Uninstall 'oai-gnb' Helm chart] *************************

TASK [helm/uninstall : Uninstall 'oai-cuup' Helm chart] ************************

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oai-cucp' Helm chart] ************************

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oai-du1' Helm chart] *************************

TASK [helm/uninstall : Uninstall 'oai-cu1' Helm chart] *************************

TASK [helm/uninstall : Uninstall 'oai-du2' Helm chart] *************************
[WARNING]: Host 

In [ ]:
# await core.remove()

In [ ]:
# await prov.unprovision()